# Data audit and workflow
Real Mendeley v3 CSVs. Recompute filters, training support and descriptive workflow transitions. Calendar duration is not labor effort.


In [1]:
from pathlib import Path
import os, sys, json
root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(root)
if str(root) not in sys.path: sys.path.insert(0, str(root))


In [2]:
from src.data import load_data, workflow
arrivals, outcomes, audit = load_data("data/raw")
print(json.dumps(audit, indent=2))


{
  "raw_issues": 66691,
  "D1_period": 13788,
  "D2_done": 12572,
  "D3_done_categories": 11961,
  "unmatched_snapshot_ids": 0,
  "missing_first_assignee": 1028,
  "D4_done_with_assignee": 11068,
  "D5_usable": 11068,
  "negative_duration": 0,
  "all_category_arrivals": 13026,
  "category_counts": {
    "T-normal": 4824,
    "T-high": 2961,
    "T-urgent": 2009,
    "Service": 1274
  },
  "train_outcomes": 7172,
  "test_outcomes": 3896,
  "arrival_assignment_time_missing": 1028,
  "sha256": {
    "issues.csv": "eab3b8857bc19bd917e0767fb9c30b7714d5468b25c5a1c1cd71a760e9602f02",
    "issues_snapshot.csv": "89975cd7e0868408ea7b0a701758d54a439255d7814fd3aa14b198f9ea543948",
    "issues_change_history.csv": "93f6d99893bac6eb755c075dc4a3c3f655cab7c6b4e2d54a3dca0b26e4b6a3c6"
  }
}


In [3]:
from src.estimates import fit
e = fit(outcomes, outcomes, "2021-01-01", "2022-07-01")
print("Eligible pairs by category:")
print(e.eligible.sum().to_string())
print("Outcome hours quantiles:")
print(outcomes.hours.quantile([.1,.5,.9,.99]).to_string())


Eligible pairs by category:
category
T-normal    82
T-high      38
T-urgent    33
Service     40
Outcome hours quantiles:
0.10       1.511111
0.50     145.767917
0.90    1312.473972
0.99    6343.493136


In [4]:
transitions = workflow("data/raw", arrivals.id)
print(transitions.head(20).to_string(index=False))
print("Missing first assignments:", audit["missing_first_assignee"])


                 previous                     value  count
                 resolved                    closed   6778
              in_progress                   waiting   5200
                  waiting                  resolved   3080
              in_progress                  resolved   3075
                  waiting                    closed   2969
              in_progress                    closed   1469
                  waiting               in_progress    732
              in_progress                validation    675
                   closed                  reopened    631
               validation                   waiting    538
resolved_under_monitoring                    closed    522
                  waiting                validation    434
                 reopened                    closed    369
                  waiting resolved_under_monitoring    355
resolved_under_monitoring                  resolved    304
              in_progress resolved_under_monitoring    2

Final-record type/priority may differ from arrival-time values. Missing first-assignee records remain in strict demand but cannot supply observed employee outcomes. See the methodological assessment for identification limits.
